# Tarea: Ajuste por Mínimos Cuadrados en MATLAB

En este cuaderno analizaremos un conjunto de 10 puntos numéricos utilizando 5 modelos de regresión por mínimos cuadrados. Evaluaremos la calidad de cada ajuste mediante dos métricas:

* **Error Cuadrático Medio ($\text{MSE}$):**
  $$\text{MSE} = \frac{1}{m} \sum_{i=1}^{m} r_i^2$$

* **Coeficiente de Determinación ($R^2$):**
  $$R^2 = 1 - \frac{\sum r_i^2}{\sum (y_i - \bar{y})^2}$$

In [ ]:
% --- 1. DATOS DE ENTRADA Y PREPARACIÓN ---
clear; clc; close all;

% Puntos conocidos
x = [4, 4.2, 4.5, 4.7, 5.1, 5.5, 5.9, 6.3, 6.8, 7.1];
y = [102.56, 113.18, 130.11, 142.05, 167.53, 195.14, 224.87, 256.73, 299.5, 326.72];

m = length(x);       % Cantidad total de puntos (10)
y_bar = y - mean(y); % Diferencia respecto a la media de y

% Puntos continuos para trazado suavizado
ps = linspace(min(x), max(x), 100);
int = interp1(x, y, ps, "spline");

% Sumatorias base para armar los sistemas matriciales
sum_x   = sum(x);         sum_y   = sum(y);
sum_x2  = sum(x.^2);     sum_xy  = sum(x.*y);
sum_x3  = sum(x.^3);     sum_x4  = sum(x.^4);
sum_x5  = sum(x.^5);     sum_x6  = sum(x.^6);
sum_x2y = sum((x.^2).*y);
sum_x3y = sum((x.^3).*y);

disp('Datos cargados e inicializados con éxito.');

## 2. Modelo 1: Lineal (Grado 1)

Ajuste de una recta con ecuación:
$$y = a_1 x + a_0$$

In [ ]:
% --- MODELO LINEAL ---
a0 = (sum_y*sum_x2 - sum_xy*sum_x) / (m*sum_x2 - sum_x^2); % Intersección
a1 = (m*sum_xy - sum_x*sum_y) / (m*sum_x2 - sum_x^2);      % Pendiente

p1 = a1.*x + a0;                        % Predicciones
r1 = y - p1;                            % Residuos
MSE1 = sum(r1.^2) / m;                  % Error Cuadrático Medio
R21 = 1 - sum(r1.^2) / sum(y_bar.^2);   % Coeficiente R^2

fprintf('Lineal: y = %.4fx + (%.4f) | MSE = %.4f | R^2 = %.4f\n', a1, a0, MSE1, R21);

## 3. Modelo 2: Polinomial de Grado 2 (Cuadrático)

Ajuste de una parábola:
$$y = a_2 x^2 + a_1 x + a_0$$

Se resuelve el sistema $A_2 \cdot \vec{a} = B_2$ usando la función `linsolve`.

In [ ]:
% --- MODELO CUADRÁTICO ---
A2 = [m,      sum_x,  sum_x2;
      sum_x,  sum_x2, sum_x3;
      sum_x2, sum_x3, sum_x4];
B2 = [sum_y; sum_xy; sum_x2y];

sol2 = linsolve(A2, B2);

p2 = sol2(1) + sol2(2).*x + sol2(3).*(x.^2);
r2 = y - p2;
MSE2 = sum(r2.^2) / m;
R22 = 1 - sum(r2.^2) / sum(y_bar.^2);

fprintf('Grado 2: y = %.4fx^2 + (%.4f)x + (%.4f) | MSE = %.4f | R^2 = %.4f\n', sol2(3), sol2(2), sol2(1), MSE2, R22);

## 4. Modelo 3: Polinomial de Grado 3 (Cúbico)

Ajuste de un polinomio de tercer grado:
$$y = a_3 x^3 + a_2 x^2 + a_1 x + a_0$$

In [ ]:
% --- MODELO CÚBICO ---
A3 = [m,      sum_x,  sum_x2, sum_x3;
      sum_x,  sum_x2, sum_x3, sum_x4;
      sum_x2, sum_x3, sum_x4, sum_x5;
      sum_x3, sum_x4, sum_x5, sum_x6];
B3 = [sum_y; sum_xy; sum_x2y; sum_x3y];

sol3 = linsolve(A3, B3);

p3 = sol3(1) + sol3(2).*x + sol3(3).*(x.^2) + sol3(4).*(x.^3);
r3 = y - p3;
MSE3 = sum(r3.^2) / m;
R23 = 1 - sum(r3.^2) / sum(y_bar.^2);

fprintf('Grado 3: y = %.4fx^3 + (%.4f)x^2 + (%.4f)x + (%.4f) | MSE = %.4f | R^2 = %.4f\n', sol3(4), sol3(3), sol3(2), sol3(1), MSE3, R23);

## 5. Modelo 4: Semilog (Exponencial)

Modelo de la forma:
$$y = a \cdot e^{bx}$$

Se linealiza aplicando logaritmo natural $\ln(y)$ y se recuperan los parámetros originales mediante la función exponencial `exp()`.

In [ ]:
% --- MODELO SEMILOG ---
ln_y = log(y);
sum_lny = sum(ln_y);
sum_xlny = sum(x.*ln_y);

A4 = [m,     sum_x;
      sum_x, sum_x2];
B4 = [sum_lny; sum_xlny];

sol4 = linsolve(A4, B4);

p4 = exp(sol4(1) + sol4(2).*x); 
r4 = y - p4;
MSE4 = sum(r4.^2) / m;
R24 = 1 - sum(r4.^2) / sum(y_bar.^2);

fprintf('Semilog: y = %.4f * e^(%.4fx) | MSE = %.4f | R^2 = %.4f\n', exp(sol4(1)), sol4(2), MSE4, R24);

## 6. Modelo 5: Log-Log (Potencial)

Modelo de la forma:
$$y = a \cdot x^b$$

Se linealiza aplicando logaritmo natural a ambas variables: $\ln(y) = \ln(a) + b \ln(x)$.

In [ ]:
% --- MODELO LOG-LOG ---
ln_x = log(x);
sum_lnx = sum(ln_x);
sum_lnx2 = sum(ln_x.^2);
sum_lnxlny = sum(ln_x.*ln_y);

A5 = [m,       sum_lnx;
      sum_lnx, sum_lnx2];
B5 = [sum_lny; sum_lnxlny];

sol5 = linsolve(A5, B5);

p5 = exp(sol5(1) + sol5(2).*ln_x);
r5 = y - p5;
MSE5 = sum(r5.^2) / m;
R25 = 1 - sum(r5.^2) / sum(y_bar.^2);

fprintf('Log-Log: y = %.4f * x^(%.4f) | MSE = %.4f | R^2 = %.4f\n', exp(sol5(1)), sol5(2), MSE5, R25);

## 7. Gráfica y Conclusiones

In [ ]:
% --- VISUALIZACIÓN CONTINUA Y MEJORADA DE RESULTADOS ---

% 1. Malla continua de 200 puntos en el rango de x
ps = linspace(min(x), max(x), 200);

% 2. Evaluación analítica continua de cada modelo
ys_p1 = a1 .* ps + a0;                                            % Lineal
ys_p2 = sol2(1) + sol2(2).*ps + sol2(3).*(ps.^2);                 % Cuadrático
ys_p3 = sol3(1) + sol3(2).*ps + sol3(3).*(ps.^2) + sol3(4).*(ps.^3); % Cúbico
ys_p4 = exp(sol4(1) + sol4(2).*ps);                               % Semilog (Exponencial)
ys_p5 = exp(sol5(1) + sol5(2).*log(ps));                         % Log-Log (Potencial)

% 3. Creación de la gráfica con lienzo blanco
figure('Color', 'w');

% Dibujamos primero los puntos reales como asteriscos destacados
scatter(x, y, 80, 'k', 'filled', 'DisplayName', 'Datos Reales');
hold on;

% Dibujamos las curvas continuas con colores y estilos bien diferenciados
plot(ps, ys_p1, 'r--', 'LineWidth', 1.5, 'DisplayName', 'Grado 1 (Lineal)');
plot(ps, ys_p2, 'b-',  'LineWidth', 2.2, 'DisplayName', 'Grado 2 (Cuadrático)');
plot(ps, ys_p3, 'm-.', 'LineWidth', 1.5, 'DisplayName', 'Grado 3 (Cúbico)');
plot(ps, ys_p4, 'g--', 'LineWidth', 1.8, 'DisplayName', 'Semilog (Exponencial)');
plot(ps, ys_p5, 'c:',  'LineWidth', 2.0, 'DisplayName', 'Log-Log (Potencial)');

% 4. Ajustes estéticos y de ejes para resaltar diferencias
grid on;
grid minor; % Cuadrícula fina para ver diferencias milimétricas

xlabel('Variable X', 'FontSize', 11, 'FontWeight', 'bold');
ylabel('Variable Y', 'FontSize', 11, 'FontWeight', 'bold');
title('Comparación de Modelos por Mínimos Cuadrados (Curvas Continuas)', 'FontSize', 12);
legend('Location', 'northwest', 'FontSize', 9);

% Encuadre exacto alrededor de los datos para ampliar la vista de las curvas
xlim([min(x)-0.2, max(x)+0.2]);
ylim([min(y)-15, max(y)+15]);

% 5. Tabla resumen de métricas numéricas en la consola
fprintf('=====================================================\n');
fprintf('         TABLA COMPARATIVA DE AJUSTE NUMÉRICO        \n');
fprintf('=====================================================\n');
fprintf('1. Grado 1 (Lineal):     MSE = %8.4f | R^2 = %.4f\n', MSE1, R21);
fprintf('2. Grado 2 (Cuadrático): MSE = %8.4f | R^2 = %.4f\n', MSE2, R22);
fprintf('3. Grado 3 (Cúbico):     MSE = %8.4f | R^2 = %.4f\n', MSE3, R23);
fprintf('4. Semilog (Exponencial):MSE = %8.4f | R^2 = %.4f\n', MSE4, R24);
fprintf('5. Log-Log (Potencial):  MSE = %8.4f | R^2 = %.4f\n', MSE5, R25);
fprintf('=====================================================\n');

Analizando los valores obtenidos de Error Cuadrático Medio ($\text{MSE}$):

* **Ajuste Lineal y Semilog:** Muestran errores elevados ($\text{MSE} = 32.9013$ y $\text{MSE} = 41.7691$ respectivamente), lo cual indica que la tendencia de los datos no es rectilínea ni exponencial pura.
* **Ajuste Log-Log:** Reduce significativamente el error ($\text{MSE} = 0.0007$) y nos revela un exponente $b \approx 2.0195$, lo que apunta a un comportamiento parabólico subyacente.
* **Polinomio de Grado 2:** Es el **modelo óptimo**. Alcanza un error insignificante ($\text{MSE} \approx 0.0001$) ofreciendo el mejor balance entre precisión y simplicidad matemática.
* **Polinomio de Grado 3:** Aunque presenta un $\text{MSE}$ numéricamente idéntico o ligeramente menor ($\approx 0.00005$), el coeficiente cúbico $a_3 = -0.0137$ es despreciable. Agregar un grado extra añade complejidad sin aportar una mejora real en el ajuste.
